In [1]:
# =============================================================================
# Step 5: Dataset Partition (v6 - No leakage)
# =============================================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

print("Step 5 Started - Using v4 Features (leakage-fixed)")

Mounted at /content/drive
Step 5 Started - Using v4 Features (leakage-fixed)


In [2]:
# ====================== 5.1 LOAD v4 FEATURES ======================
PATH = "/content/drive/MyDrive/LaDe/LaDe-P/features_engineered_v4.parquet"
df = pd.read_parquet(PATH)

print(f"Loaded v4: {df.shape[0]:,} rows × {df.shape[1]} cols")

Loaded v4: 1,132,429 rows × 48 cols


In [3]:
# ====================== 5.2 CLEANING ======================
p99 = df["eta_minutes"].quantile(0.99)
df_clean = df[df["eta_minutes"] <= p99].copy()

print(f"After outlier removal: {df_clean.shape[0]:,} rows")

After outlier removal: 1,121,151 rows


In [4]:
# ====================== 5.3 FEATURE LISTS (đồng bộ v4, không leak) ======================
# Lưu ý: courier_eta_mean / courier_eta_std / courier_dist_mean KHÔNG nằm trong num_cols ở đây
# vì sẽ được tính lại từ train_df sau khi split (mục 5.5b), tránh leak từ val/test.

num_cols_base = [
    'lat', 'lng', 'distance_km', 'estimated_travel_time',
    'accept_hour', 'accept_minute', 'day_of_week',
    'hour_sin', 'hour_cos',
    'time_to_window_start', 'time_window_duration',
    'aoi_frequency', 'courier_frequency',
    'distance_x_hour'
]

cat_cols = [
    'region_id', 'aoi_id', 'aoi_type',
    'is_weekend', 'is_peak_hour', 'is_morning_rush', 'is_evening_rush',
    'is_long_haul', 'is_processing_dominant', 'gps_both'
]

target_col = "eta_log"
id_cols = ['courier_id']  # cần giữ lại để tính courier stats sau split, sẽ bỏ khỏi feature_cols cuối

print(f"Base numerical: {len(num_cols_base)} | Categorical: {len(cat_cols)}")

Base numerical: 14 | Categorical: 10


In [5]:
# ====================== 5.4 TRAIN/VAL/TEST SPLIT 60/20/20 ======================
keep_cols = num_cols_base + cat_cols + id_cols + [target_col, "eta_minutes"]
model_df = df_clean[keep_cols].copy()

train_val_df, test_df = train_test_split(model_df, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(train_val_df, test_size=0.25, random_state=42)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

Train: 672,690 | Val: 224,230 | Test: 224,231


In [6]:
# ====================== 5.5 TARGET ENCODING (categorical, đúng cách - chỉ từ train) ======================
global_mean = train_df[target_col].mean()

te_maps = {}
for col in cat_cols:
    te_maps[col] = train_df.groupby(col)[target_col].mean()

def apply_te(d):
    d = d.copy()
    for col, mapping in te_maps.items():
        d[f"{col}_te"] = d[col].map(mapping).fillna(global_mean)
    return d

train_df = apply_te(train_df)
val_df   = apply_te(val_df)
test_df  = apply_te(test_df)

In [7]:
# ====================== 5.5b COURIER STATS (tính từ train_df only - fix leak) ======================
courier_stats = train_df.groupby('courier_id').agg(
    courier_eta_mean=('eta_minutes', 'mean'),
    courier_eta_std=('eta_minutes', 'std'),
    courier_dist_mean=('distance_km', 'mean')
).reset_index()

# Global fallback cho courier chưa từng xuất hiện ở train (cold-start)
global_eta_mean = train_df['eta_minutes'].mean()
global_eta_std  = train_df['eta_minutes'].std()
global_dist_mean = train_df['distance_km'].mean()

def apply_courier_stats(d):
    d = d.merge(courier_stats, on='courier_id', how='left')
    d['courier_eta_mean'] = d['courier_eta_mean'].fillna(global_eta_mean)
    d['courier_eta_std']  = d['courier_eta_std'].fillna(global_eta_std)
    d['courier_dist_mean'] = d['courier_dist_mean'].fillna(global_dist_mean)
    return d

train_df = apply_courier_stats(train_df)
val_df   = apply_courier_stats(val_df)
test_df  = apply_courier_stats(test_df)

# Báo cáo % courier cold-start ở val/test (courier chưa từng thấy ở train)
train_couriers = set(train_df['courier_id'].unique())
val_coldstart = (~val_df['courier_id'].isin(train_couriers)).mean() * 100
test_coldstart = (~test_df['courier_id'].isin(train_couriers)).mean() * 100
print(f"Cold-start courier — Val: {val_coldstart:.2f}% | Test: {test_coldstart:.2f}%")

Cold-start courier — Val: 0.04% | Test: 0.04%


In [8]:
# ====================== 5.6 KMEANS REFIT (chỉ trên train, như cũ - đã đúng) ======================
coord_scaler = StandardScaler()
kmeans = KMeans(n_clusters=60, random_state=42, n_init=10)

train_coords = coord_scaler.fit_transform(train_df[['lat','lng']])
val_coords   = coord_scaler.transform(val_df[['lat','lng']])
test_coords  = coord_scaler.transform(test_df[['lat','lng']])

train_df['coord_cluster_v2'] = kmeans.fit_predict(train_coords)
val_df['coord_cluster_v2']   = kmeans.predict(val_coords)
test_df['coord_cluster_v2']  = kmeans.predict(test_coords)

cluster_te = train_df.groupby('coord_cluster_v2')[target_col].mean()
train_df['coord_cluster_v2_te'] = train_df['coord_cluster_v2'].map(cluster_te).fillna(global_mean)
val_df['coord_cluster_v2_te']   = val_df['coord_cluster_v2'].map(cluster_te).fillna(global_mean)
test_df['coord_cluster_v2_te']  = test_df['coord_cluster_v2'].map(cluster_te).fillna(global_mean)

In [9]:
# ====================== 5.7 FINAL FEATURES ======================
num_cols_final = num_cols_base + ['courier_eta_mean', 'courier_eta_std', 'courier_dist_mean']
final_features = num_cols_final + [f"{c}_te" for c in cat_cols] + ['coord_cluster_v2_te']

X_train = train_df[final_features]
y_train = train_df[target_col]
y_train_min = train_df["eta_minutes"]

X_val = val_df[final_features]
y_val = val_df[target_col]
y_val_min = val_df["eta_minutes"]

X_test = test_df[final_features]
y_test = test_df[target_col]
y_test_min = test_df["eta_minutes"]

print(f"Final feature count: {len(final_features)}")
print(final_features)

Final feature count: 28
['lat', 'lng', 'distance_km', 'estimated_travel_time', 'accept_hour', 'accept_minute', 'day_of_week', 'hour_sin', 'hour_cos', 'time_to_window_start', 'time_window_duration', 'aoi_frequency', 'courier_frequency', 'distance_x_hour', 'courier_eta_mean', 'courier_eta_std', 'courier_dist_mean', 'region_id_te', 'aoi_id_te', 'aoi_type_te', 'is_weekend_te', 'is_peak_hour_te', 'is_morning_rush_te', 'is_evening_rush_te', 'is_long_haul_te', 'is_processing_dominant_te', 'gps_both_te', 'coord_cluster_v2_te']


In [10]:
# ====================== 5.8 SAVE ARTEFACTS ======================
CKPT_DIR = "/content/drive/MyDrive/LaDe/Checkpoints_v2"   # thư mục mới, tránh ghi đè checkpoint cũ bị leak
os.makedirs(CKPT_DIR, exist_ok=True)

joblib.dump(X_train, f"{CKPT_DIR}/X_train.pkl")
joblib.dump(X_val,   f"{CKPT_DIR}/X_val.pkl")
joblib.dump(X_test,  f"{CKPT_DIR}/X_test.pkl")

joblib.dump(y_train, f"{CKPT_DIR}/y_train.pkl")
joblib.dump(y_val,   f"{CKPT_DIR}/y_val.pkl")
joblib.dump(y_test,  f"{CKPT_DIR}/y_test.pkl")

joblib.dump(y_train_min, f"{CKPT_DIR}/y_train_min.pkl")
joblib.dump(y_val_min,   f"{CKPT_DIR}/y_val_min.pkl")
joblib.dump(y_test_min,  f"{CKPT_DIR}/y_test_min.pkl")

joblib.dump(final_features, f"{CKPT_DIR}/final_features.pkl")
joblib.dump(te_maps, f"{CKPT_DIR}/te_maps.pkl")
joblib.dump(courier_stats, f"{CKPT_DIR}/courier_stats.pkl")

print("Step 5 v6 (no-leakage) completed and artefacts saved!")

Step 5 v6 (no-leakage) completed and artefacts saved!
